<a href="https://colab.research.google.com/github/DrJamesThomas/Ai_Models/blob/main/ML_RAG_Medical_Manual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install nbconvert

In [ ]:
!jupyter nbconvert --clear-output --inplace "/content/drive/MyDrive/Notebooks/JamesThomas2RAG.ipynb"

[NbConvertApp] WARNING | pattern '/content/drive/MyDrive/Notebooks/JamesThomas2RAG.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--Jupy

In [ ]:
!pip install pandas langchain langchain-community sentence-transformers chromadb llama-cpp-python PyMuPDF huggingface-hub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 26.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import json
import pandas as pd

# LangChain utilities
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

# Hugging Face & LLaMA
from huggingface_hub import hf_hub_download
from llama_cpp import Llama


In [ ]:
model_path = hf_hub_download(
    repo_id="TheBloke/Mistral-7B-Instruct-v0.1-GGUF",
    filename="mistral-7b-instruct-v0.1.Q4_K_M.gguf" # add filename
)

llm = Llama(
    model_path=model_path,
    n_ctx=2048,
    n_threads=4
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


mistral-7b-instruct-v0.1.Q4_K_M.gguf:   0%|          | 0.00/4.37G [00:00<?, ?B/s]

llama_model_loader: loaded meta data with 20 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.1-GGUF/snapshots/731a9fc8f06f5f5e2db8a0cf9d256197eb6e05d1/mistral-7b-instruct-v0.1.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.1
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv

In [ ]:

def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

In [ ]:
queries = [ "What is the protocol for managing sepsis in a critical care unit?" ,
           "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
           "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
           " What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
           "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
]


In [ ]:
print("LLM Based Answers")
for ques in queries:
  print("\nQ:", ques)
  print("A:", response(ques))
  print("-"*60)




LLM Based Answers

Q: What is the protocol for managing sepsis in a critical care unit?


llama_perf_context_print:        load time =    2783.67 ms
llama_perf_context_print: prompt eval time =    2781.44 ms /    16 tokens (  173.84 ms per token,     5.75 tokens per second)
llama_perf_context_print:        eval time =   71214.32 ms /   127 runs   (  560.74 ms per token,     1.78 tokens per second)
llama_perf_context_print:       total time =   74068.78 ms /   143 tokens
llama_perf_context_print:    graphs reused =        122
Llama.generate: 2 prefix-match hit, remaining 32 prompt tokens to eval


A: 

Sepsis is a life-threatening condition that requires prompt and aggressive management in a critical care unit. The protocol for managing sepsis in a critical care unit typically involves the following steps:

1. Early recognition: Sepsis should be suspected in any patient who is critically ill and has a suspected or confirmed infection.
2. Rapid diagnosis: Blood cultures should be obtained as soon as possible to identify the causative pathogen.
3. Appropriate antibiotic therapy: Antibiotics should be started as soon as the causative pathogen is identified.
4.
------------------------------------------------------------

Q: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?


llama_perf_context_print:        load time =    2783.67 ms
llama_perf_context_print: prompt eval time =    7630.01 ms /    32 tokens (  238.44 ms per token,     4.19 tokens per second)
llama_perf_context_print:        eval time =       0.00 ms /     1 runs   (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:       total time =    7631.05 ms /    33 tokens
llama_perf_context_print:    graphs reused =          0
Llama.generate: 4 prefix-match hit, remaining 34 prompt tokens to eval


A: 
------------------------------------------------------------

Q: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?


...

A: 


Sudden patchy hair loss, also known as alopecia areata, is a common hair loss condition that affects both men and women. It is characterized by the sudden appearance of small, round or oval-shaped bald spots on the scalp. The exact cause of alopecia areata is not fully understood, but it is believed to be an autoimmune disorder that affects the hair follicles.

There are several effective treatments and solutions for addressing sudden patchy hair loss, including:

1. Topical corticosteroids: These are anti-inflammat
------------------------------------------------------------

Q:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?


llama_perf_context_print:        load time =    2783.67 ms
llama_perf_context_print: prompt eval time =    8256.32 ms /    30 tokens (  275.21 ms per token,     3.63 tokens per second)
llama_perf_context_print:        eval time =   70349.24 ms /   127 runs   (  553.93 ms per token,     1.81 tokens per second)
llama_perf_context_print:       total time =   78676.67 ms /   157 tokens
llama_perf_context_print:    graphs reused =        122
Llama.generate: 1 prefix-match hit, remaining 36 prompt tokens to eval


A: 


There are several treatments that may be recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function. These treatments may include:

1. Medical care: The first step in treating a brain injury is to receive medical care. This may include emergency medical care, such as surgery or medication, to stabilize the condition.

2. Rehabilitation: Rehabilitation is a critical component of brain injury treatment. It may include physical therapy, occupational therapy, speech therapy, and cognitive rehabilitation.

3. Medications: Dep
------------------------------------------------------------

Q: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?


llama_perf_context_print:        load time =    2783.67 ms
llama_perf_context_print: prompt eval time =    6006.99 ms /    36 tokens (  166.86 ms per token,     5.99 tokens per second)
llama_perf_context_print:        eval time =   70342.82 ms /   127 runs   (  553.88 ms per token,     1.81 tokens per second)
llama_perf_context_print:       total time =   76420.78 ms /   163 tokens
llama_perf_context_print:    graphs reused =        122


A: 



1. First aid: The first step is to stabilize the leg and prevent further injury. This can be done by applying a splint or cast to the leg. If the fracture is severe, it may be necessary to use crutches or a walker to help the person move around.

2. Medical attention: It is important to seek medical attention as soon as possible. A doctor will assess the severity of the fracture and determine the best course of treatment. This may involve surgery to realign the broken bone, or physical therapy to help the person regain strength and flexibility.

------------------------------------------------------------


In [ ]:
def prompt_response(query, style, temp):
  prompt=f"[INST] {style} {query} [/INST]"
  return response(prompt, temperature=temp)

styles = [ "Answer briefy:",
          "Explain like a medical expert:",
          "Provide step by step clinical protocol:",
          "Explain with different examples:",
          "Give detailed explanation in a beginner friendly way:"
]

temperatures = [0.2, 0.7, 0.5, 1.0, 1.2]

print("Prompt Engineering Results")

for style in styles:
  for temp in temperatures:
    print("\n STYLE:", style)
    print("TEMPERATURE:", temp)
    print(prompt_response(queries[0], style, temp))
    print("-"*60)


Prompt Engineering Results

 STYLE: Answer briefy:
TEMPERATURE: 0.2


Llama.generate: 1 prefix-match hit, remaining 26 prompt tokens to eval
llama_perf_context_print:        load time =    2783.67 ms
llama_perf_context_print: prompt eval time =    5150.29 ms /    26 tokens (  198.09 ms per token,     5.05 tokens per second)
llama_perf_context_print:        eval time =   63084.28 ms /   113 runs   (  558.27 ms per token,     1.79 tokens per second)
llama_perf_context_print:       total time =   68297.74 ms /   139 tokens
llama_perf_context_print:    graphs reused =        108
Llama.generate: 26 prefix-match hit, remaining 1 prompt tokens to eval


 The protocol for managing sepsis in a critical care unit typically involves a rapid identification and treatment of the infection, along with close monitoring of vital signs and organ function. This may include administering antibiotics, fluid resuscitation, and supportive care such as mechanical ventilation or vasopressors. The specific treatment and management may vary depending on the severity of the sepsis and the underlying medical conditions of the patient. It's important to note that sepsis is a medical emergency and prompt and appropriate management is crucial for improving outcomes.
------------------------------------------------------------

 STYLE: Answer briefy:
TEMPERATURE: 0.7


llama_perf_context_print:        load time =    2783.67 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   49977.13 ms /    90 runs   (  555.30 ms per token,     1.80 tokens per second)
llama_perf_context_print:       total time =   50026.21 ms /    91 tokens
llama_perf_context_print:    graphs reused =         86
Llama.generate: 26 prefix-match hit, remaining 1 prompt tokens to eval


 The protocol for managing sepsis in a critical care unit typically involves early recognition and rapid response to prevent severe complications. This may include administering antibiotics, providing fluid and electrolyte support, and using vasopressors to maintain blood pressure. Close monitoring of vital signs, laboratory values, and tissue oxygenation is also crucial. The specific management may vary depending on the severity of sepsis and the underlying causes.
------------------------------------------------------------

 STYLE: Answer briefy:
TEMPERATURE: 0.5


...

 Sepsis is a serious medical condition that occurs when the body's response to infection causes damage to its own tissues and organs. In a critical care unit, sepsis is managed using a protocol that involves a set of guidelines and procedures to ensure that the patient receives the best possible care.

The first step in managing sepsis in a critical care unit is to identify the patient as being at high risk of developing sepsis. This is done by evaluating the patient's medical history, vital signs, and laboratory tests.

Once a patient is identified as being at high risk of seps
------------------------------------------------------------

 STYLE: Give detailed explanation in a beginner friendly way:
TEMPERATURE: 1.0


llama_perf_context_print:        load time =    2783.67 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   72302.25 ms /   128 runs   (  564.86 ms per token,     1.77 tokens per second)
llama_perf_context_print:       total time =   72376.57 ms /   129 tokens
llama_perf_context_print:    graphs reused =        124
Llama.generate: 32 prefix-match hit, remaining 1 prompt tokens to eval


 Sepsis is a life-threatening condition that occurs when the body's response to infection causes inflammation and organ damage. In a critical care unit, managing sepsis requires a coordinated effort from multiple healthcare professionals, including doctors, nurses, and other medical staff.

The protocol for managing sepsis in a critical care unit typically involves the following steps:

1. Early recognition: Sepsis can be difficult to diagnose, so it's important for healthcare professionals to be aware of the signs and symptoms. These can include fever, chills, rapid breathing or heartbeat, and
------------------------------------------------------------

 STYLE: Give detailed explanation in a beginner friendly way:
TEMPERATURE: 1.2


llama_perf_context_print:        load time =    2783.67 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   72110.61 ms /   128 runs   (  563.36 ms per token,     1.78 tokens per second)
llama_perf_context_print:       total time =   72183.68 ms /   129 tokens
llama_perf_context_print:    graphs reused =        124


 Sepsis is a serious condition that can occur when the body's immune system responds to an infection in a way that causes more harm than good. It can be life-threatening if not managed properly, which is why critical care units (ICUs) have specific protocols in place for managing sepsis.

Here's a general overview of the protocol for managing sepsis in a critical care unit:

1. Early recognition and diagnosis: The first step in managing sepsis is to recognize it early and diagnose it quickly. This involves monitoring patients for signs and symptoms of seps
------------------------------------------------------------


In [ ]:
# uncomment and run the following lines for Google Colab
from google.colab import drive
drive.mount('/content/drive')
loader = PyMuPDFLoader("/content/drive/MyDrive/Notebooks/medical_diagnosis_manual.pdf")
documents = loader.load()
print("Total documents loaded:", len(documents))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total documents loaded: 4114


In [ ]:
#Data Overview
#Checking the first 5 pages

print("First 5 pages")
for i in range(5):
  print(f"\nPage {i+1}:")
  print(documents[i].page_content[:500])


First 5 pages

Page 1:
jamesothomas3@gmail.com
4E2X8AMD70
This file is meant for personal use by jamesothomas3@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.

Page 2:
jamesothomas3@gmail.com
4E2X8AMD70
This file is meant for personal use by jamesothomas3@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.

Page 3:
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .................................

Page 4:
491
Chapter 44. Foot & Ankle Disorders    .........................................

In [ ]:

print("Total number of pages:", len(documents))

Total number of pages: 4114


In [ ]:

#Data Chunking

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap= 20
)
docs = text_splitter.split_documents(documents)
print("Total chunks created:", len(docs))


Total chunks created: 8463


In [ ]:

print("Sample Chunk")
print(docs[200].page_content)


Sample Chunk
risks in people with implanted defibrillators is unclear.
Underwater (hydrostatic) weighing is the most accurate method for measuring percentage of body fat.
Costly and time-consuming, it is used more often in research than in clinical care. To be weighed
accurately while submerged, people must fully exhale beforehand.
Imaging procedures, including CT, MRI, and dual-energy x-ray absorptiometry (DEXA), can also estimate
the percentage and distribution of body fat but are usually used only for research.
Other testing: Obese patients should be screened for obstructive sleep apnea with an instrument such
as the Epworth Sleepiness Scale and often the apnea-hypopnea index (total number of apnea or
hypopnea episodes occurring per hour of sleep—see p. 1904). This disorder is often underdiagnosed,
and obesity increases the risk.
Fasting glucose and lipid levels should be measured routinely in patients with a large waist circumference
or a family history of type 2 diabetes mellitus 

In [ ]:
#Embedding
embedding_model = SentenceTransformerEmbeddings(
  model_name = "sentence-transformers/all-MiniLM-L6-v2"
)


/tmp/ipykernel_28796/818503462.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
embedding_1 = embedding_model.embed_query(docs[0].page_content)
embedding_2 = embedding_model.embed_query(docs[1].page_content)

In [ ]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  384


True

In [ ]:
out_dir = 'MedManual'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)


In [ ]:

vectorstore = Chroma.from_documents(
    docs,
    embedding_model,
    persist_directory=out_dir
)


In [ ]:
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

vectorstore.embeddings

/tmp/ipykernel_28796/455608138.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)


HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [ ]:

vectorstore.similarity_search("Underwater (hydrostatic)",k=3)

[Document(metadata={'source': '/content/drive/MyDrive/Notebooks/medical_diagnosis_manual.pdf', 'moddate': '2026-03-16T03:53:24+00:00', 'total_pages': 4114, 'author': '', 'format': 'PDF 1.7', 'creator': 'Atop CHM to PDF Converter', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'modDate': 'D:20260316035324Z', 'file_path': '/content/drive/MyDrive/Notebooks/medical_diagnosis_manual.pdf', 'creationDate': 'D:20120615054440Z', 'creationdate': '2012-06-15T05:44:40+00:00', 'page': 3477, 'trapped': '', 'keywords': '', 'subject': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition'}, page_content='and sputum Gram stain and culture.\nPrognosis\nFactors that increase the chance of surviving submersion without permanent injury include the following:\n• Brief duration of submersion\n• Cold water temperature\n• Young age\n• Absence of underlying medical conditions, secondary trauma, and aspiration of particulate matter or\nchemicals\n• Rapid institution of resuscitation (

In [ ]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 2}
)

In [ ]:
rel_docs = retriever.invoke("How to treat Sepsis?")

In [ ]:
rel_docs

[Document(metadata={'author': '', 'source': '/content/drive/MyDrive/Notebooks/medical_diagnosis_manual.pdf', 'moddate': '2026-03-16T03:53:24+00:00', 'creationdate': '2012-06-15T05:44:40+00:00', 'total_pages': 4114, 'format': 'PDF 1.7', 'creationDate': 'D:20120615054440Z', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'keywords': '', 'file_path': '/content/drive/MyDrive/Notebooks/medical_diagnosis_manual.pdf', 'page': 2456, 'subject': '', 'creator': 'Atop CHM to PDF Converter', 'modDate': 'D:20260316035324Z', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'trapped': ''}, page_content="Parenteral antibiotics should be given after specimens of blood, body fluids, and wound sites have been\ntaken for Gram stain and culture. Very prompt empiric therapy, started immediately after suspecting\nsepsis, is essential and may be lifesaving. Antibiotic selection requires an educated guess based on the\nsuspected source, clinical setting, knowledge or suspicion of ca

In [ ]:
# System and User Prompt Template
# 1. The system message describing the assistant's role.
# 2. A user message template including context and the question.

qna_system_message = """
You are an assistant whose work is to review the report and provide the appropriate answers from the context.
User input will have the context required by you to answer user questions.
This context will begin with the token: ###Context.
The context contains references to specific portions of a document relevant to the user query.

User questions will begin with the token: ###Question.

Please answer only using the context provided in the input. Do not mention anything about the context in your final answer.

If the answer is not found in the context, respond "I don't know".
"""

In [ ]:
qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""

In [ ]:
# Response Function
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.invoke(user_input)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

In [ ]:
# Question Answering using RAG
user_input = "What is the protocol for managing sepsis in a critical care unit?"
print(generate_rag_response(user_input))

Llama.generate: 1 prefix-match hit, remaining 1315 prompt tokens to eval
llama_perf_context_print:        load time =    2783.67 ms
llama_perf_context_print: prompt eval time =  267978.07 ms /  1315 tokens (  203.79 ms per token,     4.91 tokens per second)
llama_perf_context_print:        eval time =   77133.52 ms /   127 runs   (  607.35 ms per token,     1.65 tokens per second)
llama_perf_context_print:       total time =  345182.11 ms /  1442 tokens
llama_perf_context_print:    graphs reused =        122


###Answer
Sepsis is a life-threatening condition that requires prompt and aggressive management in a critical care unit. The protocol for managing sepsis includes the following steps:

1. First aid: Keep the patient warm, control hemorrhage, check the airway and ventilation, and provide supplemental oxygen by face mask.
2. Treatment: Intubate the airway with mechanical ventilation if necessary, and insert two large IV catheters into separate peripheral veins. A central venous line or an intraosseous needle may be used if peripheral


In [ ]:
# Question 2
user_input = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(generate_rag_response(user_input))

Llama.generate: 145 prefix-match hit, remaining 897 prompt tokens to eval
llama_perf_context_print:        load time =    2809.63 ms
llama_perf_context_print: prompt eval time =  189300.87 ms /   897 tokens (  211.04 ms per token,     4.74 tokens per second)
llama_perf_context_print:        eval time =   74272.48 ms /   127 runs   (  584.82 ms per token,     1.71 tokens per second)
llama_perf_context_print:       total time =  263647.26 ms /  1024 tokens
llama_perf_context_print:    graphs reused =        122


###Answer
The common symptoms for appendicitis are epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Additional signs are pain felt in the right lower quadrant with palpation of the left lower quad


In [ ]:
# Question 3
user_input = " What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(generate_rag_response(user_input))

Llama.generate: 145 prefix-match hit, remaining 670 prompt tokens to eval
llama_perf_context_print:        load time =    2809.63 ms
llama_perf_context_print: prompt eval time =  143473.30 ms /   670 tokens (  214.14 ms per token,     4.67 tokens per second)
llama_perf_context_print:        eval time =   80443.21 ms /   127 runs   (  633.41 ms per token,     1.58 tokens per second)
llama_perf_context_print:       total time =  223991.33 ms /   797 tokens
llama_perf_context_print:    graphs reused =        122


###Answer
 The possible causes behind sudden patchy hair loss, commonly seen as localized bald spots on the scalp, include alopecia areata, which is an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers, such as stress, trauma, or certain medications. Other possible causes include fungal infections, vitamin deficiencies, and hormonal imbalances.

Effective treatments for alopecia areata include topical corticosteroids, retinoids, or immunosuppressants, which can help reduce inflammation


In [ ]:
# Question 4
user_input = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(generate_rag_response(user_input))

Llama.generate: 145 prefix-match hit, remaining 524 prompt tokens to eval
llama_perf_context_print:        load time =    2809.63 ms
llama_perf_context_print: prompt eval time =  115250.41 ms /   524 tokens (  219.94 ms per token,     4.55 tokens per second)
llama_perf_context_print:        eval time =   70753.20 ms /   126 runs   (  561.53 ms per token,     1.78 tokens per second)
llama_perf_context_print:       total time =  186075.55 ms /   650 tokens
llama_perf_context_print:    graphs reused =        121


###Answer
The treatment recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function is supportive care. This includes preventing systemic complications due to immobilization, providing good nutrition, and preventing pressure ulcers. There is no specific treatment for brain damage. Support groups may also provide assistance to the families of brain-injured patients. For patients whose coma exceeds 24 hours, a prolonged period of rehabilitation, particularly in cognitive and emotional areas, is often required. Rehabilitation services should be planned early.


In [ ]:
# Question 5
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(generate_rag_response(user_input))

Llama.generate: 145 prefix-match hit, remaining 1160 prompt tokens to eval
llama_perf_context_print:        load time =    2809.63 ms
llama_perf_context_print: prompt eval time =  233869.42 ms /  1160 tokens (  201.61 ms per token,     4.96 tokens per second)
llama_perf_context_print:        eval time =   72501.19 ms /   127 runs   (  570.88 ms per token,     1.75 tokens per second)
llama_perf_context_print:       total time =  306441.35 ms /  1287 tokens
llama_perf_context_print:    graphs reused =        122


###Answer
For a person who has fractured their leg during a hiking trip, the necessary precautions and treatment steps include:

1. Immobilization: The leg should be immobilized immediately by splinting to prevent further injury to soft tissues and to decrease pain.
2. Pain management: Pain should be treated typically with opioids.
3. Definitive treatment: Definitive treatment often involves reduction, which usually requires analgesia or sedation. Closed reduction is done when possible; if not, open reduction is done.
4. Rehabilitation:


Fine Tuning

In [ ]:
user_input = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input, max_tokens=100)

Llama.generate: 145 prefix-match hit, remaining 1160 prompt tokens to eval
llama_perf_context_print:        load time =    2809.63 ms
llama_perf_context_print: prompt eval time =  238006.77 ms /  1160 tokens (  205.18 ms per token,     4.87 tokens per second)
llama_perf_context_print:        eval time =   57531.78 ms /    99 runs   (  581.13 ms per token,     1.72 tokens per second)
llama_perf_context_print:       total time =  295591.59 ms /  1259 tokens
llama_perf_context_print:    graphs reused =         95


'###Answer\nFor a person who has fractured their leg during a hiking trip, the necessary precautions and treatment steps include:\n\n1. Immobilization: The leg should be immobilized immediately by splinting to prevent further injury to soft tissues and to decrease pain.\n2. Pain management: Pain should be treated typically with opioids.\n3. Definitive treatment: Definitive treatment often involves reduction, which usually requires analges'

In [ ]:
user_input_2 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_2, temperature=0.1, max_tokens=150)



Llama.generate: 1304 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    2809.63 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   94336.28 ms /   150 runs   (  628.91 ms per token,     1.59 tokens per second)
llama_perf_context_print:       total time =   94428.88 ms /   151 tokens
llama_perf_context_print:    graphs reused =        144


'###Answer\nFor a person who has fractured their leg during a hiking trip, the necessary precautions and treatment steps include:\n\n1. Immobilization: The leg should be immobilized immediately by splinting to prevent further injury to soft tissues and to decrease pain.\n2. Pain management: Pain should be treated typically with opioids.\n3. Definitive treatment: Definitive treatment often involves reduction, which usually requires analgesia or sedation. Closed reduction is done when possible; if not, open reduction is done.\n4. Rehabilitation: Rehabilitation is started as soon as possible after hip fracture surgery. The first goals may be to increase'

In [ ]:
user_input_3 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_3, top_p=0.98, top_k=10, max_tokens=200)


Llama.generate: 1304 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    2809.63 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  120273.65 ms /   200 runs   (  601.37 ms per token,     1.66 tokens per second)
llama_perf_context_print:       total time =  120406.26 ms /   201 tokens
llama_perf_context_print:    graphs reused =        193


'###Answer\nFor a person who has fractured their leg during a hiking trip, the necessary precautions and treatment steps include:\n\n1. Immobilization: The leg should be immobilized immediately by splinting to prevent further injury to soft tissues and to decrease pain.\n2. Pain management: Pain should be treated typically with opioids.\n3. Definitive treatment: Definitive treatment often involves reduction, which usually requires analgesia or sedation. Closed reduction is done when possible; if not, open reduction is done.\n4. Rehabilitation: Rehabilitation is started as soon as possible after hip fracture surgery. The first goals may be to increase strength and to prevent atrophy on the unaffected side. Initially, only isometric exercise of the affected limb while it is fully extended is permitted. Gradual mobilization of the affected limb usually results in full ambulation.'

In [ ]:
user_input_4 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_4, top_p=0.98, top_k=15, max_tokens=256)


Llama.generate: 1304 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    2809.63 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  153588.48 ms /   256 runs   (  599.96 ms per token,     1.67 tokens per second)
llama_perf_context_print:       total time =  153776.19 ms /   257 tokens
llama_perf_context_print:    graphs reused =        247


'###Answer\nFor a person who has fractured their leg during a hiking trip, the necessary precautions and treatment steps include:\n\n1. Immobilization: The leg should be immobilized immediately by splinting to prevent further injury to soft tissues and to decrease pain.\n2. Pain management: Pain should be treated typically with opioids.\n3. Definitive treatment: Definitive treatment often involves reduction, which usually requires analgesia or sedation. Closed reduction is done when possible; if not, open reduction is done.\n4. Rehabilitation: Rehabilitation is started as soon as possible after hip fracture surgery. The first goals may be to increase strength and to prevent atrophy on the unaffected side. Initially, only isometric exercise of the affected limb while it is fully extended is permitted. Gradual mobilization of the affected limb usually results in full ambulation.\n\nFor their care and recovery, the person should:\n\n1. Follow the treatment plan prescribed by the healthcar

In [ ]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_5, top_p=0.98, top_k=20, max_tokens=350)


Llama.generate: 1304 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    2809.63 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =  212474.98 ms /   350 runs   (  607.07 ms per token,     1.65 tokens per second)
llama_perf_context_print:       total time =  212770.45 ms /   351 tokens
llama_perf_context_print:    graphs reused =        338


'###Answer\nFor a person who has fractured their leg during a hiking trip, the necessary precautions and treatment steps include:\n\n1. Immobilization: The leg should be immobilized immediately by splinting to prevent further injury to soft tissues and to decrease pain.\n2. Pain management: Pain should be treated typically with opioids.\n3. Definitive treatment: Definitive treatment often involves reduction, which usually requires analgesia or sedation. Closed reduction is done when possible; if not, open reduction is done.\n4. Rehabilitation: Rehabilitation is started as soon as possible after hip fracture surgery. The first goals may be to increase strength and to prevent atrophy on the unaffected side. Initially, only isometric exercise of the affected limb while it is fully extended is permitted. Gradual mobilization of the affected limb usually results in full ambulation.\n\nFor their care and recovery, the person should:\n\n1. Follow the treatment plan prescribed by the healthcar

Output Evaluation

In [ ]:
groundedness_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score.
"""

In [ ]:
relevance_rater_system_message = """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.
Consider whether all and only the important aspects are contained in the answer when evaluating relevance.

Instructions:
1. First write down the steps that are needed to evaluate the context as per the metric.
2. Give a step-by-step explanation if the context adheres to the metric considering the question as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the context using the evaluaton criteria and assign a score.
"""


In [ ]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

Evaluation Function

In [ ]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=1024,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.invoke(user_input)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]
    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

In [ ]:
# Query 1
user_input = "What is the protocol for managing sepsis in a critical care unit?"
ground,rel = generate_ground_relevance_response(user_input,max_tokens=350)

print(ground,end="\n\n")
print(rel)

Llama.generate: 1 prefix-match hit, remaining 1330 prompt tokens to eval
llama_perf_context_print:        load time =    2809.63 ms
llama_perf_context_print: prompt eval time =  303842.55 ms /  1330 tokens (  228.45 ms per token,     4.38 tokens per second)
llama_perf_context_print:        eval time =  190128.99 ms /   321 runs   (  592.30 ms per token,     1.69 tokens per second)
llama_perf_context_print:       total time =  494233.86 ms /  1651 tokens
llama_perf_context_print:    graphs reused =        310
Llama.generate: 7 prefix-match hit, remaining 1771 prompt tokens to eval
llama_perf_context_print:        load time =    2809.63 ms
llama_perf_context_print: prompt eval time =  357819.64 ms /  1771 tokens (  202.04 ms per token,     4.95 tokens per second)
llama_perf_context_print:        eval time =   91412.97 ms /   153 runs   (  597.47 ms per token,     1.67 tokens per second)
llama_perf_context_print:       total time =  449324.11 ms /  1924 tokens
llama_perf_context_print:   

 Steps to evaluate the answer:

1. Read the question and context to understand the topic and the information provided.
2. Identify the key points that need to be included in the answer based on the context.
3. Check if the answer includes all the key points identified in step 2.
4. Evaluate the extent to which the answer adheres to the metric by checking if it is derived only from the information presented in the context.

Evaluation:

The answer adheres to the metric to a good extent as it includes all the key points identified in step 2 and is derived only from the information presented in the context.

Rating: 4 - The metric is followed mostly.

 1. To evaluate the context as per the metric, the following steps can be taken:
* Identify the main aspects of the question: In this case, the main aspects of the question are the protocol for managing sepsis in a critical care unit, the steps involved in managing sepsis, and the importance of fluid therapy and antibiotics in managing sepsi

In [ ]:
#Query 2
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
ground,rel = generate_ground_relevance_response(user_input_2,max_tokens=400)

print(ground,end="\n\n")
print(rel)

Llama.generate: 7 prefix-match hit, remaining 1050 prompt tokens to eval
llama_perf_context_print:        load time =    2809.63 ms
llama_perf_context_print: prompt eval time =  204617.43 ms /  1050 tokens (  194.87 ms per token,     5.13 tokens per second)
llama_perf_context_print:        eval time =  153916.50 ms /   267 runs   (  576.47 ms per token,     1.73 tokens per second)
llama_perf_context_print:       total time =  358729.84 ms /  1317 tokens
llama_perf_context_print:    graphs reused =        258
Llama.generate: 7 prefix-match hit, remaining 1443 prompt tokens to eval
llama_perf_context_print:        load time =    2809.63 ms
llama_perf_context_print: prompt eval time =  281699.57 ms /  1443 tokens (  195.22 ms per token,     5.12 tokens per second)
llama_perf_context_print:        eval time =   96324.53 ms /   166 runs   (  580.27 ms per token,     1.72 tokens per second)
llama_perf_context_print:       total time =  378125.26 ms /  1609 tokens
llama_perf_context_print:   

 Steps to evaluate the answer:

1. Read the question and context carefully.
2. Identify the key points that the AI system should consider while generating the answer.
3. Evaluate the extent to which the AI system has followed the key points.
4. Rate the answer using the evaluation criteria.

Evaluation:

The AI system has followed the key points mentioned in the question and context to a good extent. The answer clearly states the common symptoms of appendicitis and confirms that it cannot be cured via medicine. It also provides the surgical procedure to treat it. However, the answer could have been more comprehensive by providing more details about the surgical removal procedure and its potential complications.

Rating: 4 - The metric is followed mostly.

 Steps to evaluate the context as per the metric:

1. Identify the main aspects of the question: The main aspects of the question are the common symptoms of appendicitis and whether it can be cured via medicine.
2. Check if all and on

In [ ]:
#Query 3
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
ground,rel = generate_ground_relevance_response(user_input_3,max_tokens=500)

print(ground,end="\n\n")
print(rel)

Llama.generate: 1 prefix-match hit, remaining 829 prompt tokens to eval
llama_perf_context_print:        load time =    2783.67 ms
llama_perf_context_print: prompt eval time =  161452.02 ms /   829 tokens (  194.76 ms per token,     5.13 tokens per second)
llama_perf_context_print:        eval time =  208104.21 ms /   348 runs   (  598.00 ms per token,     1.67 tokens per second)
llama_perf_context_print:       total time =  369840.28 ms /  1177 tokens
llama_perf_context_print:    graphs reused =        336
Llama.generate: 7 prefix-match hit, remaining 1297 prompt tokens to eval
llama_perf_context_print:        load time =    2783.67 ms
llama_perf_context_print: prompt eval time =  253756.09 ms /  1297 tokens (  195.65 ms per token,     5.11 tokens per second)
llama_perf_context_print:        eval time =  182838.61 ms /   301 runs   (  607.44 ms per token,     1.65 tokens per second)
llama_perf_context_print:       total time =  436819.62 ms /  1598 tokens
llama_perf_context_print:    

 Steps to evaluate the answer as per the metric:

1. Read the question and context carefully to understand the information presented.
2. Check if the answer is derived only from the information presented in the context.
3. Evaluate the extent to which the answer adheres to the metric.
4. Rate the answer using the evaluation criteria.

Step-by-step explanation if the answer adheres to the metric:

1. The question asks for effective treatments or solutions for addressing sudden patchy hair loss and possible causes behind it.
2. The context provides information on nonscarring alopecia, etiology of alopecias, and possible causes of hair loss.
3. The AI generated answer lists the effective treatments for alopecia areata and possible causes of sudden patchy hair loss.
4. The answer is derived only from the information presented in the context, as it does not include any additional or irrelevant information.

Evaluation of the extent to which the metric is followed:

The answer adheres to the

In [ ]:
#Query 4
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
ground,rel = generate_ground_relevance_response(user_input_4,max_tokens=300)
print(ground,end="\n\n")
print(rel)

Llama.generate: 7 prefix-match hit, remaining 677 prompt tokens to eval
llama_perf_context_print:        load time =    2783.67 ms
llama_perf_context_print: prompt eval time =  131289.09 ms /   677 tokens (  193.93 ms per token,     5.16 tokens per second)
llama_perf_context_print:        eval time =   72485.97 ms /   124 runs   (  584.56 ms per token,     1.71 tokens per second)
llama_perf_context_print:       total time =  203846.46 ms /   801 tokens
llama_perf_context_print:    graphs reused =        119
Llama.generate: 7 prefix-match hit, remaining 927 prompt tokens to eval
llama_perf_context_print:        load time =    2783.67 ms
llama_perf_context_print: prompt eval time =  181457.61 ms /   927 tokens (  195.75 ms per token,     5.11 tokens per second)
llama_perf_context_print:        eval time =  184409.13 ms /   299 runs   (  616.75 ms per token,     1.62 tokens per second)
llama_perf_context_print:       total time =  366093.10 ms /  1226 tokens
llama_perf_context_print:    g

 Steps to evaluate the answer as per the metric:

1. Read the question and context carefully to understand the information presented.
2. Determine if the answer is derived only from the information presented in the context.
3. Evaluate the extent to which the answer adheres to the metric.
4. Rate the answer using the evaluation criteria and assign a score.

Step-by-step explanation if the answer adheres to the metric:

1. The question asks for the recommended treatments for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function.
2. The context provides information about the severity of brain damage, the likelihood of recovery, and the recommended treatments.
3. The AI generated answer is derived only from the information presented in the context and does not include any additional or irrelevant information.
4. The answer adheres to the metric as it is derived solely from the information presented in the context.

R

In [ ]:
#Query 5
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
ground,rel = generate_ground_relevance_response(user_input_5,max_tokens=200)
print(ground,end="\n\n")
print(rel)

Llama.generate: 7 prefix-match hit, remaining 1313 prompt tokens to eval
llama_perf_context_print:        load time =    2783.67 ms
llama_perf_context_print: prompt eval time =  262928.12 ms /  1313 tokens (  200.25 ms per token,     4.99 tokens per second)
llama_perf_context_print:        eval time =  122563.23 ms /   199 runs   (  615.90 ms per token,     1.62 tokens per second)
llama_perf_context_print:       total time =  385620.90 ms /  1512 tokens
llama_perf_context_print:    graphs reused =        192
Llama.generate: 7 prefix-match hit, remaining 1639 prompt tokens to eval
llama_perf_context_print:        load time =    2783.67 ms
llama_perf_context_print: prompt eval time =  343852.04 ms /  1639 tokens (  209.79 ms per token,     4.77 tokens per second)
llama_perf_context_print:        eval time =   78263.79 ms /   124 runs   (  631.16 ms per token,     1.58 tokens per second)
llama_perf_context_print:       total time =  422187.48 ms /  1763 tokens
llama_perf_context_print:   

 Steps to evaluate the answer:

1. Check if the answer is derived only from the information presented in the context.
2. Check if the answer adheres to the metric by comparing it with the question and context.
3. Rate the answer using the evaluation criteria based on the extent to which the metric is followed.

Evaluation:

The answer adheres to the metric to a good extent as it is derived only from the information presented in the context and it provides a comprehensive answer to the question.

Rating: 4 - The metric is followed mostly.

 To evaluate the context as per the metric, the following steps can be taken:

1. Identify the important aspects of the question: In this case, the important aspects of the question are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery.
2. Check if all and only the important aspects are contained in the context: The context provides inf

Business Insights and Recommendations

The RAG system successfully implemented a local, quantized
Mistral-7B-GGUF model to perform complex information retrieval on a 4,114-page medical manual.

The document was broken into approximately 8,400 chunks, embedded using all-MiniLM-L6-v2, and stored in a Chroma vector store.1

Testing showed high accuracy and trustworthiness.

Evaluation scores for groundedness were consistently 4 or 5, indicating the LLM reliably answered questions using only the provided context and minimized hallucinations.

The Relevance Rater confirmed the retriever (k=2 or k=3) was effective at finding specific clinical protocols for niche medical questions.1

Parameter tuning was critical: a temperature setting of 0.0 was found to be the most effective for medical Q&A, ensuring deterministic and safe responses. Formatting was improved by using [INST] tags to structure clear clinical protocols.1

The key limitation observed was high latency (100-300+ seconds per query) caused by local CPU-based inference, suggesting that GPU acceleration is necessary for production use.

Future optimization includes increasing the retrieval depth (k) from 2 to 5 to improve comprehensiveness for complex surgical queries.